In [ ]:
import time
import random
import sys
import tracemalloc


def knapsack_memoization(weights, values, capacity):
    """
    Solve the 0/1 Knapsack Problem using memoization (top-down recursion).

    Args:
        weights:  List of item weights (non-negative integers).
        values:   List of item values   (non-negative integers).
        capacity: Maximum weight capacity (non-negative integer).

    Returns:
        The maximum possible value that fits within the capacity.

    Raises:
        ValueError: If inputs are invalid.
    """
    if not isinstance(weights, (list, tuple)) or not isinstance(values, (list, tuple)):
        raise TypeError("weights and values must be lists or tuples.")
    if not isinstance(capacity, (int, float)):
        raise TypeError("capacity must be an integer or float.")

    weights = list(weights)
    values = list(values)

    if len(weights) != len(values):
        raise ValueError("weights and values must have the same length.")
    if any(w < 0 for w in weights):
        raise ValueError("All weights must be non-negative.")
    if any(v < 0 for v in values):
        raise ValueError("All values must be non-negative.")
    if capacity < 0:
        raise ValueError("Capacity must be non-negative.")

    n = len(weights)

    if n == 0 or capacity == 0:
        return 0

    memo = {}

    def _solve(idx, remaining):
        """
        Recursive helper that returns the max value obtainable
        using items idx..n-1 with 'remaining' weight capacity.
        """
        if idx == n or remaining == 0:
            return 0

        state = (idx, remaining)
        if state in memo:
            return memo[state]

        exclude = _solve(idx + 1, remaining)

        include = 0
        if weights[idx] <= remaining:
            include = values[idx] + _solve(idx + 1, remaining - weights[idx])

        memo[state] = max(include, exclude)
        return memo[state]

    return _solve(0, capacity)


def knapsack_tabulation(weights, values, capacity):
    """
    Solve the 0/1 Knapsack Problem using tabulation (bottom-up DP).

    Args:
        weights:  List of item weights (non-negative integers).
        values:   List of item values   (non-negative integers).
        capacity: Maximum weight capacity (non-negative integer).

    Returns:
        The maximum possible value that fits within the capacity.

    Raises:
        ValueError: If inputs are invalid.
    """
    if not isinstance(weights, (list, tuple)) or not isinstance(values, (list, tuple)):
        raise TypeError("weights and values must be lists or tuples.")
    if not isinstance(capacity, (int, float)):
        raise TypeError("capacity must be an integer or float.")

    weights = list(weights)
    values = list(values)

    if len(weights) != len(values):
        raise ValueError("weights and values must have the same length.")
    if any(w < 0 for w in weights):
        raise ValueError("All weights must be non-negative.")
    if any(v < 0 for v in values):
        raise ValueError("All values must be non-negative.")
    if capacity < 0:
        raise ValueError("Capacity must be non-negative.")

    n = len(weights)

    if n == 0 or capacity == 0:
        return 0

    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        w_i = weights[i - 1]
        v_i = values[i - 1]
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]

            if w_i <= w:
                dp[i][w] = max(dp[i][w], v_i + dp[i - 1][w - w_i])

    return dp[n][capacity]


def analyze_implementations():
    """
    Measure and compare the performance of both implementations
    across a range of input sizes.
    """

    def _benchmark(func, weights, values, capacity, warmup_runs=1, runs=5):
        """
        Run *func* multiple times and return (elapsed_seconds, peak_memory_bytes).
        """
        for _ in range(warmup_runs):
            func(weights, values, capacity)

        times = []
        peak_memory = 0

        for _ in range(runs):
            tracemalloc.start()
            start = time.perf_counter()
            func(weights, values, capacity)
            end = time.perf_counter()
            tracemalloc.stop()

            elapsed = end - start
            times.append(elapsed)

            _, current_peak = tracemalloc.get_traced_memory()
            if current_peak > peak_memory:
                peak_memory = current_peak

        avg_time = sum(times) / len(times)
        return avg_time, peak_memory

    print("=" * 72)
    print("  0/1 KNAPSACK — MEMOIZATION vs. TABULATION  |  PERFORMANCE ANALYSIS")
    print("=" * 72)

    test_cases = [
        {
            "name": "Basic",
            "weights": [2, 3, 4, 5],
            "values":  [3, 4, 5, 6],
            "capacity": 8,
        },
        {
            "name": "Empty input",
            "weights": [],
            "values":  [],
            "capacity": 10,
        },
        {
            "name": "Capacity = 0",
            "weights": [2, 3, 4],
            "values":  [3, 4, 5],
            "capacity": 0,
        },
        {
            "name": "Single item (fits)",
            "weights": [5],
            "values":  [10],
            "capacity": 5,
        },
        {
            "name": "Single item (too heavy)",
            "weights": [10],
            "values":  [10],
            "capacity": 5,
        },
        {
            "name": "All items too heavy",
            "weights": [10, 20, 30],
            "values":  [5, 10, 15],
            "capacity": 5,
        },
    ]

    sizes = [50, 100, 200, 500]

    for n in sizes:
        random.seed(42) 
        w = [random.randint(1, 100) for _ in range(n)]
        v = [random.randint(1, 100) for _ in range(n)]
        cap = random.randint(n * 20, n * 60)
        test_cases.append({
            "name": f"Random ({n} items)",
            "weights": w,
            "values":  v,
            "capacity": cap,
        })

    results = []

    for tc in test_cases:
        name = tc["name"]
        w = tc["weights"]
        v = tc["values"]
        c = tc["capacity"]

        print(f"\n--- Test: {name}  (n={len(w)}, capacity={c}) ---")

        try:
            t1, m1 = _benchmark(knapsack_memoization, w, v, c)
            r1 = knapsack_memoization(w, v, c)
            print(f"  Memoization  →  result={r1}  |  avg time={t1:.6f}s  |  peak mem={m1 / 1024:.1f} KB")
        except RecursionError:
            r1 = "RecursionError"
            t1, m1 = None, None
            print(f"  Memoization  →  RecursionError (depth exceeded)")
        except Exception as e:
            r1 = str(e)
            t1, m1 = None, None
            print(f"  Memoization  →  Error: {e}")

        try:
            t2, m2 = _benchmark(knapsack_tabulation, w, v, c)
            r2 = knapsack_tabulation(w, v, c)
            print(f"  Tabulation   →  result={r2}  |  avg time={t2:.6f}s  |  peak mem={m2 / 1024:.1f} KB")
        except Exception as e:
            r2 = str(e)
            t2, m2 = None, None
            print(f"  Tabulation   →  Error: {e}")

        if r1 != "RecursionError" and r1 != "Error" and r2 != "Error" and r1 != r2:
            print(f"  ⚠  WARNING: Results differ! memo={r1}, tab={r2}")

        results.append({
            "name": name,
            "n": len(w),
            "capacity": c,
            "memo_time": t1,
            "memo_mem": m1,
            "memo_result": r1,
            "tab_time": t2,
            "tab_mem": m2,
            "tab_result": r2,
        })

    print("\n" + "=" * 72)
    print("  SUMMARY")
    print("=" * 72)
    header = f"{'Test':<25} {'n':>4} {'Memo(s)':>10} {'Tab(s)':>10} {'Memo(MB)':>9} {'Tab(MB)':>9} {'Match?':>7}"
    print(header)
    print("-" * len(header))

    for r in results:
        mt = f"{r['memo_time']:.6f}" if r['memo_time'] is not None else "N/A"
        tt = f"{r['tab_time']:.6f}" if r['tab_time'] is not None else "N/A"
        mm = f"{r['memo_mem'] / 1024 / 1024:.2f}" if r['memo_mem'] else "N/A"
        tm = f"{r['tab_mem'] / 1024 / 1024:.2f}" if r['tab_mem'] else "N/A"
        match = "✓" if (r['memo_result'] == r['tab_result']) or \
                       (r['memo_result'] == "RecursionError" and r['tab_result'] == "RecursionError") else "✗"
        print(f"{r['name']:<25} {r['n']:>4} {mt:>10} {tt:>10} {mm:>9} {tm:>9} {match:>7}")

    print("=" * 72)


if __name__ == "__main__":
    analyze_implementations()
